<a href="https://colab.research.google.com/github/HeYantong/MSSP6070-Yantong/blob/main/Assignments/Participation_Activity_Week9_HeYantong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess

github_token = userdata.get("Git_Key")

owner = "HeYantong"
repository = "MSSP6070-Yantong"
repository_path = Path("/content") / repository

github_token = userdata.get("Git_Key")

if not github_token:
    raise ValueError(
        "Git_Key was not found in Colab Secrets."
    )

if not (repository_path / ".git").exists():

    clone_url = (
        f"https://x-access-token:{github_token}"
        f"@github.com/{owner}/{repository}.git"
    )

    subprocess.run(
        [
            "git",
            "clone",
            clone_url,
            str(repository_path)
        ],
        check=True
    )

os.chdir(repository_path)
print("Current directory:", Path.cwd())

Current directory: /content/MSSP6070-Yantong


In [24]:
import time
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

DIRECTORY_URL = "https://sp2.upenn.edu/sp2-directory/"
OUTPUT_DIR = Path("/content/sp2_faculty_output")
IMAGE_DIR = OUTPUT_DIR / "faculty_images"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "MSSP6070 educational web-scraping assignment; respectful low-rate requests"
})

def get_soup(url):
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

print("Output folder:", OUTPUT_DIR)
print("Image folder:", IMAGE_DIR)

Output folder: /content/sp2_faculty_output
Image folder: /content/sp2_faculty_output/faculty_images


In [26]:
def get_label_links(section, label_name):
    """Extract links under Affiliation or Program(s)."""

    if section is None:
        return []

    label = section.find(
        "span",
        class_="sidebar-label",
        string=lambda text: (
            text
            and text.strip() == label_name
        )
    )

    paragraph = (
        label.find_next_sibling("p")
        if label
        else None
    )

    if paragraph is None:
        return []

    return [
        link.get_text(" ", strip=True)
        for link in paragraph.select("a")
    ]

In [27]:
# Download the first directory page
first_soup = get_soup(DIRECTORY_URL)

# Find the total number of directory pages
page_numbers = [
    int(tag["data-page"])
    for tag in first_soup.select("[data-page]")
    if tag.get("data-page", "").isdigit()
]

max_page = max(page_numbers, default=1)

# Use a set to prevent duplicate profile URLs
profile_urls = set()

for page in range(1, max_page + 1):

    page_url = (
        DIRECTORY_URL
        if page == 1
        else urljoin(
            DIRECTORY_URL,
            f"page/{page}/"
        )
    )

    soup = (
        first_soup
        if page == 1
        else get_soup(page_url)
    )

    for link in soup.select(
        "a.card-link-wrapper[href]"
    ):
        profile_url = urljoin(
            page_url,
            link["href"]
        )

        profile_urls.add(profile_url)

    print(
        f"Directory page {page}/{max_page}"
    )

    if page < max_page:
        time.sleep(0.5)

# Convert the set into a sorted list
profile_urls = sorted(profile_urls)

print("Number of profiles found:",len(profile_urls))

Directory page 1/28
Directory page 2/28
Directory page 3/28
Directory page 4/28
Directory page 5/28
Directory page 6/28
Directory page 7/28
Directory page 8/28
Directory page 9/28
Directory page 10/28
Directory page 11/28
Directory page 12/28
Directory page 13/28
Directory page 14/28
Directory page 15/28
Directory page 16/28
Directory page 17/28
Directory page 18/28
Directory page 19/28
Directory page 20/28
Directory page 21/28
Directory page 22/28
Directory page 23/28
Directory page 24/28
Directory page 25/28
Directory page 26/28
Directory page 27/28
Directory page 28/28
Number of profiles found: 330


In [35]:
faculty_records = []

for number, profile_url in enumerate(profile_urls, start=1):

    try:
        soup = get_soup(profile_url)

        about_section = None

        for section in soup.select(".faculty-sidebar__section"):
            heading = section.select_one("h2.sidebar-title")

            if heading and heading.get_text(strip=True) == "About":
                about_section = section
                break

        affiliations = get_label_links(
            about_section,
            "Affiliation"
        )

        if "Faculty" not in affiliations:
            continue

        name = soup.select_one("h1.faculty-hero__name")
        title = soup.select_one(".faculty-hero__subheading")
        email = soup.select_one('a[href^="mailto:"]')
        image = soup.select_one("img.faculty-hero__image")

        programs = get_label_links(
            about_section,
            "Program(s)"
        )

        research_interests = []

        if about_section:
            research_interests = [
                item.get_text(" ", strip=True)
                for item in about_section.select(
                    ".faculty-about-section li"
                )
            ]

        biography = None

        biography_heading = soup.find(
            "h2",
            string=lambda text: text and text.strip() == "Biography"
        )

        if biography_heading:
            biography_body = biography_heading.find_next_sibling(
                "div",
                class_="entry-content"
            )

            if biography_body:
                biography = biography_body.get_text(
                    " ",
                    strip=True
                )

        image_url = None

        if image and image.get("src"):
            image_url = urljoin(
                profile_url,
                image["src"]
            )

        image_file = None

        if image_url:

            try:
                image_response = session.get(
                    image_url,
                    timeout=30
                )

                image_response.raise_for_status()

                content_type = image_response.headers.get(
                    "Content-Type",
                    ""
                )

                if content_type.startswith("image/"):

                    slug = Path(
                        urlparse(
                            profile_url
                        ).path.rstrip("/")
                    ).name

                    extension = Path(
                        urlparse(image_url).path
                    ).suffix or ".jpg"

                    image_path = (
                        IMAGE_DIR
                        / f"{slug}{extension}"
                    )

                    image_path.write_bytes(
                        image_response.content
                    )

                    image_file = str(image_path)

            except Exception as image_error:
                print(
                    "Image download failed:",
                    profile_url,
                    image_error
                )

        faculty_records.append({
            "name": name.get_text(" ", strip=True) if name else None,
            "title": title.get_text(" ", strip=True) if title else None,
            "email": email.get_text(strip=True) if email else None,
            "affiliations": "; ".join(affiliations),
            "programs": "; ".join(programs),
            "research_interests": "; ".join(research_interests),
            "biography": biography,
            "profile_url": profile_url,
            "image_url": image_url,
            "image_file": image_file
        })

        print(
            f"{number}/{len(profile_urls)}:",
            faculty_records[-1]["name"]
        )

    except Exception as error:
        print("Error:", profile_url, error)

    time.sleep(0.5)

2/330: Abigail (Abby) M. Ross, LICSW, MPH, PhD
6/330: Adam Roth-Saks, MSEd
12/330: Alice Xu, PhD
14/330: Alisa G. Shockley, PhD
16/330: Allan Irving, MSW, PhD
18/330: Allison Werner-Lin, PhD, LCSW
24/330: Amy Hillier, MSW, PhD
29/330: Andrew Lamas, JD
35/330: Antoine Germain, PhD
38/330: Ariel Schwartz, MPA, PhD
40/330: Aviv Y. Landau, PhD, MSW
43/330: Barrie K. Levine, MSW
46/330: Benjamin Todd Jealous
47/330: Beth A. Clauss, MSW, LCSW
48/330: Beth Hallowell, PhD
49/330: Bonnie Scranton, DSW, LCSW, MSW
52/330: Bruce Boylston, CPA
53/330: Bruce Warren, MS
56/330: Candice Dias, PhD
58/330: Caren Kelman, MSW
61/330: Carol Wilson Spigner, MSW, DSW
62/330: Caroline Menapace Glavin, DSW, LCSW
63/330: M. Caroline Wong, MSW
65/330: Casey McGovern, MSW
66/330: Cassie Haynes, JD, MPH
67/330: Catherine (Cate) Revak, LSW
69/330: Celeste Vaughan-Briggs, LCSW
70/330: Chao Guo, PhD
71/330: Charles Johnson, PhD
72/330: Charu Juneja, MBA
75/330: ChiaKo Hung
78/330: Christian Talbot, MA
86/330: Dana Ka

In [37]:
import shutil
from google.colab import files

# Convert the collected records into a DataFrame
faculty_df = pd.DataFrame(
    faculty_records
)

# Add the official photo URL
faculty_df["photo_link"] = faculty_df["image_url"]

# Keep only the image filename in the CSV
faculty_df["image_filename"] = faculty_df[
    "image_file"
].apply(
    lambda path: (
        Path(path).name
        if isinstance(path, str) and path
        else None
    )
)

# Save the faculty data as a CSV file
csv_path = OUTPUT_DIR / "sp2_faculty.csv"

faculty_df.to_csv(
    csv_path,
    index=False
)

# Compress all downloaded images
photos_zip = shutil.make_archive(
    str(OUTPUT_DIR / "faculty_images"),
    "zip",
    root_dir=IMAGE_DIR
)

print("\nCompleted!")
print(
    "Number of faculty members:",
    len(faculty_df)
)
print("CSV file:", csv_path)
print("Photos ZIP file:", photos_zip)

# Preview selected columns
display(
    faculty_df[
        [
            "name",
            "title",
            "email",
            "programs",
            "research_interests",
            "photo_link",
            "image_filename"
        ]
    ].head()
)

# Download the CSV file and the image ZIP file
files.download(str(csv_path))
files.download(photos_zip)


Completed!
Number of faculty members: 155
CSV file: /content/sp2_faculty_output/sp2_faculty.csv
Photos ZIP file: /content/sp2_faculty_output/faculty_images.zip


,name,title,email,programs,research_interests,photo_link,image_filename
0,"Abigail (Abby) M. Ross, LICSW, MPH, PhD",Associate Professor,amross@upenn.edu,DSW; MSSP; MSW; PhD,Intervention research; Implementation science;...,https://sp2.upenn.edu/wp-content/uploads/2026/...,abigail-m-ross.jpg
1,"Adam Roth-Saks, MSEd",None,adamsaks@upenn.edu,NPL,,https://sp2.upenn.edu/wp-content/uploads/2026/...,adam-roth-saks-msed.jpg
2,"Alice Xu, PhD",Assistant Professor,alicezxu@upenn.edu,MSSP; PhD,Inequality and social policy; Urban and distri...,https://sp2.upenn.edu/wp-content/uploads/2026/...,alice-xu.jpeg
3,"Alisa G. Shockley, PhD","Lecturer, MSSP Program",ashocks@upenn.edu,MSSP,,https://sp2.upenn.edu/wp-content/uploads/2026/...,alisa-g-shockley.jpg
4,"Allan Irving, MSW, PhD","Lecturer, MSW Program",allani@upenn.edu,MSSP; MSW,,https://sp2.upenn.edu/wp-content/uploads/2026/...,allan-irving.jpg


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>